# FINAL Evaluation Model Training - A100 Optimized

**Hardware:** Colab Pro A100 (40GB) + High RAM

**Features:**
- 6 datasets (SciQ, GSM8K, OpenBookQA, ARC-Easy, ARC-Challenge, CommonsenseQA, ELI5)
- Optuna hyperparameter optimization
- A100-optimized settings (larger batches, more workers)
- ~50K+ training examples

**Expected time:** ~6-7 hours total (3-4h Optuna + 3-4h final training)

## 1. Setup & Install

In [1]:
# Install packages
!pip install -q datasets transformers accelerate peft bitsandbytes trl torch optuna joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 37.9 MB/s eta 0:00:00


In [2]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!nvidia-smi

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
Sun Feb 22 10:29:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             52W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |          

In [3]:
# Setup Kaggle
import os
os.environ['KAGGLE_USERNAME'] = 'yahia saqer'
os.environ['KAGGLE_KEY'] = 'KGAT_9502dfc610067da2128caa08ad564633'
!pip install -q kaggle

In [4]:
# Download ELI5
print("Downloading ELI5 dataset...")
!kaggle datasets download -d trandaiphu/eli5-dataset
!unzip -q eli5-dataset.zip
print("✅ ELI5 downloaded!")

Dataset URL: https://www.kaggle.com/datasets/trandaiphu/eli5-dataset
License(s): unknown
 93% 1.83G/1.96G [00:01<00:00, 1.57GB/s]
100% 1.96G/1.96G [00:01<00:00, 1.60GB/s]
✅ ELI5 downloaded!


In [5]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Import Libraries

In [6]:
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
import random
import optuna
import joblib
from typing import List, Dict
import pandas as pd
import json
from multiprocessing import cpu_count

print(f"CPU cores: {cpu_count()}")
print("✅ Libraries loaded")

CPU cores: 12
✅ Libraries loaded


## 3. Load 6 Datasets

In [7]:
print("Loading 6 datasets...\n")

# 1. SciQ
sciq = load_dataset("allenai/sciq")
print(f"✓ SciQ: {len(sciq['train'])} train")

# 2. GSM8K
gsm8k = load_dataset("openai/gsm8k", "main")
print(f"✓ GSM8K: {len(gsm8k['train'])} train")

# 3. OpenBookQA
openbookqa = load_dataset("allenai/openbookqa", "main")
print(f"✓ OpenBookQA: {len(openbookqa['train'])} train")

# 4. ARC-Easy
arc_easy = load_dataset("allenai/ai2_arc", "ARC-Easy")
print(f"✓ ARC-Easy: {len(arc_easy['train'])} train")

# 5. ARC-Challenge
arc_challenge = load_dataset("allenai/ai2_arc", "ARC-Challenge")
print(f"✓ ARC-Challenge: {len(arc_challenge['train'])} train")

# 6. CommonsenseQA
commonsenseqa = load_dataset("tau/commonsense_qa")
print(f"✓ CommonsenseQA: {len(commonsenseqa['train'])} train")

# 7. ELI5 (limited sample)
print("\nLoading ELI5 sample...")
eli5_data_list = []
limit = 50000
with open('ELI5.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i >= limit:
            break
        eli5_data_list.append(json.loads(line))

eli5_df = pd.DataFrame(eli5_data_list)
eli5_data = eli5_df.to_dict('records')
print(f"✓ ELI5: {len(eli5_df)} examples")

print("\n" + "="*80)
print("ALL 6 DATASETS LOADED!")
print("="*80)

Loading 6 datasets...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

✓ CommonsenseQA: 9741 train

Loading ELI5 sample...
✓ ELI5: 50000 examples

ALL 6 DATASETS LOADED!


## 4. Prompt Generation Functions

In [8]:
def create_multi_question_prompt_sciq(examples: List[Dict], correct: bool, preference: str) -> str:
    context = examples[0]['support']
    qa_section = ""
    for i, ex in enumerate(examples, 1):
        question = ex['question']
        if correct:
            answer = ex['correct_answer']
        else:
            distractors = [ex['distractor1'], ex['distractor2'], ex['distractor3']]
            answer = random.choice([d for d in distractors if d]) or ex['correct_answer']
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = f"""The student demonstrated strong understanding across all {len(examples)} questions."""
        recommendation = f"""Ready to progress. Recommend: {'Next video' if preference == 'videos' else 'Next article'}."""
    else:
        evaluation = f"""The student shows misconceptions, needs to review foundational concepts."""
        recommendation = f"""Needs reinforcement. Recommend: {'Review video' if preference == 'videos' else 'Review article'}."""

    return f"""### Domain: [SCIENCE]
### Context:
{context}

### Assessment:
{qa_section}### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_gsm8k(examples: List[Dict], correct: bool, preference: str) -> str:
    context = "Mathematics lesson on word problem solving."
    qa_section = ""
    for i, ex in enumerate(examples, 1):
        question = ex['question']
        if correct:
            full_answer = ex['answer']
            if '####' in full_answer:
                reasoning = full_answer.split('####')[0].strip()
                final = full_answer.split('####')[1].strip()
                answer = f"{reasoning[:100]}... Answer: {final}"
            else:
                answer = full_answer[:100]
        else:
            if '####' in ex['answer']:
                correct_num = ex['answer'].split('####')[1].strip()
                try:
                    wrong = str(int(float(correct_num)) + random.choice([-5, -2, 2, 5]))
                    answer = f"Incorrect approach. Answer: {wrong}"
                except:
                    answer = "Incorrect approach."
            else:
                answer = "Incorrect approach."
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = f"""The student correctly solved all {len(examples)} problems with proper reasoning."""
        recommendation = f"""Ready for harder problems. Recommend: {'Next video' if preference == 'videos' else 'Next article'}."""
    else:
        evaluation = f"""The student struggled, making calculation errors or using wrong approaches."""
        recommendation = f"""Needs practice. Recommend: {'Review video' if preference == 'videos' else 'Review article'}."""

    return f"""### Domain: [MATH]
### Context:
{context}

### Assessment:
{qa_section}### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_arc(examples: List[Dict], correct: bool, preference: str) -> str:
    context = "Science reasoning and knowledge test."
    qa_section = ""
    for i, ex in enumerate(examples, 1):
        question = ex['question']
        choices = ex['choices']
        if correct:
            answer_idx = choices['label'].index(ex['answerKey'])
            answer = choices['text'][answer_idx]
        else:
            wrong_indices = [j for j in range(len(choices['text'])) if choices['label'][j] != ex['answerKey']]
            answer = choices['text'][random.choice(wrong_indices)]
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = """The student answered correctly, showing strong scientific reasoning."""
        recommendation = f"""Ready to advance. Recommend: {'Next video' if preference == 'videos' else 'Next article'}."""
    else:
        evaluation = """The student's answers suggest gaps in scientific knowledge."""
        recommendation = f"""Needs review. Recommend: {'Review video' if preference == 'videos' else 'Review article'}."""

    return f"""### Domain: [SCIENCE]
### Context:
{context}

### Assessment:
{qa_section}### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_commonsense(examples: List[Dict], correct: bool, preference: str) -> str:
    context = "Common sense reasoning test."
    qa_section = ""
    for i, ex in enumerate(examples, 1):
        question = ex['question']
        choices = ex['choices']
        if correct:
            answer_idx = choices['label'].index(ex['answerKey'])
            answer = choices['text'][answer_idx]
        else:
            wrong_indices = [j for j in range(len(choices['text'])) if choices['label'][j] != ex['answerKey']]
            answer = choices['text'][random.choice(wrong_indices)]
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = """The student demonstrated good common sense reasoning."""
        recommendation = f"""Ready to advance. Recommend: {'Next video' if preference == 'videos' else 'Next article'}."""
    else:
        evaluation = """The student needs to improve logical reasoning skills."""
        recommendation = f"""Needs practice. Recommend: {'Review video' if preference == 'videos' else 'Review article'}."""

    return f"""### Domain: [REASONING]
### Context:
{context}

### Assessment:
{qa_section}### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_openbook(examples: List[Dict], correct: bool, preference: str) -> str:
    context = examples[0].get('fact1', 'General knowledge and reasoning.')
    qa_section = ""
    for i, ex in enumerate(examples, 1):
        question = ex['question_stem']
        choices = ex['choices']
        if correct:
            answer_idx = ord(ex['answerKey']) - ord('A')
            answer = choices['text'][answer_idx]
        else:
            wrong_indices = [j for j in range(len(choices['text'])) if j != (ord(ex['answerKey']) - ord('A'))]
            answer = choices['text'][random.choice(wrong_indices)]
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = """The student answered correctly, showing strong critical thinking."""
        recommendation = f"""Ready to advance. Recommend: {'Next video' if preference == 'videos' else 'Next article'}."""
    else:
        evaluation = """The student's answers suggest difficulty with logical reasoning."""
        recommendation = f"""Needs practice. Recommend: {'Review video' if preference == 'videos' else 'Review article'}."""

    return f"""### Domain: [REASONING]
### Context:
{context}

### Assessment:
{qa_section}### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_eli5(example: Dict, correct: bool, preference: str) -> str:
    question = example['question']
    context = "Educational content explaining complex concepts."
    if correct:
        answer_data = example['answers']
        if isinstance(answer_data, list):
            answer = answer_data[0][:200] if answer_data else "No answer"
        else:
            answer = str(answer_data)[:200]
    else:
        answer = "Incomplete or unclear explanation"

    if correct:
        evaluation = """The student provided a clear explanation showing understanding."""
        recommendation = f"""Ready for more. Recommend: {'Next video' if preference == 'videos' else 'Next article'}."""
    else:
        evaluation = """The student's explanation shows gaps in understanding."""
        recommendation = f"""Needs review. Recommend: {'Review video' if preference == 'videos' else 'Review article'}."""

    return f"""### Domain: [GENERAL]
### Context:
{context}

### Assessment:
Q1: {question}
Student Answer: {answer}

### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""

print("✅ Prompt functions created")

✅ Prompt functions created


## 5. Prepare Combined Dataset

In [9]:
def prepare_combined_dataset_all(sciq_data, gsm8k_data, openbook_data, arc_easy_data,
                                  arc_challenge_data, commonsense_data, eli5_dataset_slice, split="train"):
    """Prepare combined dataset with all 6 sources"""
    all_prompts = []
    preferences = ["videos", "articles"]

    # SciQ - group 3
    print("Processing SciQ...")
    sciq_split = sciq_data[split]
    for i in range(0, min(len(sciq_split), 12000) - 2, 3):  # Increased from 10K
        examples = [sciq_split[j] for j in range(i, i+3)]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_sciq(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_sciq(examples, False, random.choice(preferences)))

    # GSM8K - group 2
    print("Processing GSM8K...")
    gsm8k_split_key = split if split in gsm8k_data else "test"
    gsm8k_split = gsm8k_data[gsm8k_split_key]
    for i in range(0, min(len(gsm8k_split), 6000) - 1, 2):  # Increased from 5K
        examples = [gsm8k_split[j] for j in range(i, i+2)]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_gsm8k(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_gsm8k(examples, False, random.choice(preferences)))

    # OpenBookQA - group 3
    print("Processing OpenBookQA...")
    openbook_split = openbook_data[split]
    for i in range(0, len(openbook_split) - 2, 3):
        examples = [openbook_split[j] for j in range(i, i+3)]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_openbook(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_openbook(examples, False, random.choice(preferences)))

    # ARC-Easy - group 3
    print("Processing ARC-Easy...")
    arc_easy_split = arc_easy_data[split]
    for i in range(0, len(arc_easy_split) - 2, 3):
        examples = [arc_easy_split[j] for j in range(i, i+3)]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_arc(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_arc(examples, False, random.choice(preferences)))

    # ARC-Challenge - group 3
    print("Processing ARC-Challenge...")
    arc_challenge_split = arc_challenge_data[split]
    for i in range(0, len(arc_challenge_split) - 2, 3):
        examples = [arc_challenge_split[j] for j in range(i, i+3)]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_arc(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_arc(examples, False, random.choice(preferences)))

    # CommonsenseQA - group 3
    print("Processing CommonsenseQA...")
    commonsense_split = commonsense_data[split]
    for i in range(0, len(commonsense_split) - 2, 3):
        examples = [commonsense_split[j] for j in range(i, i+3)]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_commonsense(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_commonsense(examples, False, random.choice(preferences)))

    # ELI5
    print("Processing ELI5...")
    for i in range(min(len(eli5_dataset_slice), 6000)):  # Increased from 5K
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_eli5(eli5_dataset_slice[i], True, pref))
        all_prompts.append(create_multi_question_prompt_eli5(eli5_dataset_slice[i], False, random.choice(preferences)))

    random.shuffle(all_prompts)
    return all_prompts

print("✅ Data preparation function ready")

✅ Data preparation function ready


In [10]:
# Prepare datasets
print("Preparing training data...")
train_prompts = prepare_combined_dataset_all(
    sciq, gsm8k, openbookqa, arc_easy, arc_challenge, commonsenseqa, eli5_data[:12000], "train"
)

print("\nPreparing validation data...")
val_prompts = prepare_combined_dataset_all(
    sciq, gsm8k, openbookqa, arc_easy, arc_challenge, commonsenseqa, eli5_data[12000:13000], "validation"
)

print(f"\n{'='*80}")
print(f"Training examples: {len(train_prompts)}")
print(f"Validation examples: {len(val_prompts)}")
print(f"{'='*80}")

train_dataset = Dataset.from_dict({"text": train_prompts})
val_dataset = Dataset.from_dict({"text": val_prompts})

print("\n✅ Datasets ready!")

Preparing training data...
Processing SciQ...
Processing GSM8K...
Processing OpenBookQA...
Processing ARC-Easy...
Processing ARC-Challenge...
Processing CommonsenseQA...
Processing ELI5...

Preparing validation data...
Processing SciQ...
Processing GSM8K...
Processing OpenBookQA...
Processing ARC-Easy...
Processing ARC-Challenge...
Processing CommonsenseQA...
Processing ELI5...

Training examples: 37830
Validation examples: 5708

✅ Datasets ready!


## 6. Optuna Hyperparameter Optimization

**A100 Optimized Settings:**
- Larger batch sizes (16, 32 vs 4, 8)
- Larger LoRA ranks (16, 32, 64 vs 8, 16, 32)
- More workers for data loading

In [11]:
model_name = "mistralai/Mistral-7B-v0.1"

def objective(trial):
    """A100-optimized Optuna objective"""

    # Hyperparameters - OPTIMIZED FOR A100
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    lora_r = trial.suggest_categorical("lora_r", [16, 32, 64])  # Larger ranks
    lora_alpha = trial.suggest_categorical("lora_alpha", [32, 64, 128])  # Larger alpha
    batch_size = trial.suggest_categorical("batch_size", [16, 32])  # MUCH larger batches for A100
    warmup_steps = trial.suggest_int("warmup_steps", 100, 400)

    print(f"\n{'='*80}")
    print(f"Trial {trial.number}")
    print(f"LR: {learning_rate:.2e}, LoRA r: {lora_r}, alpha: {lora_alpha}")
    print(f"Batch size: {batch_size}, Warmup: {warmup_steps}")
    print(f"{'='*80}\n")

    # Load model with 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # LoRA config
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = prepare_model_for_kbit_training(model)

    # Training args - A100 OPTIMIZED
    training_args = TrainingArguments(
        output_dir=f"./optuna-trial-{trial.number}",
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=1,  # No accumulation with large batches
        learning_rate=learning_rate,
        bf16=True,
        eval_strategy="steps",
        eval_steps=200,
        logging_steps=50,
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        max_grad_norm=0.3,
        save_strategy="no",
        dataloader_num_workers=4,  # A100 + High RAM = more workers
        dataloader_pin_memory=True,
        group_by_length=True,  # Faster training
    )

    # Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset.select(range(8000)),  # Larger subset for Optuna
        eval_dataset=val_dataset.select(range(800)),
        peft_config=peft_config,
        args=training_args,
        dataset_text_field="text",
        max_seq_length=1024,
        tokenizer=tokenizer,
    )

    # Train
    trainer.train()

    # Evaluate
    eval_result = trainer.evaluate()
    eval_loss = eval_result["eval_loss"]

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()

    return eval_loss

print("✅ Optuna objective function ready")

✅ Optuna objective function ready


In [19]:
# Run Optuna
print("Starting Optuna hyperparameter search...")
print("Trials: 10")
print("Estimated time: ~3-4 hours on A100\n")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("\n" + "="*80)
print("OPTUNA RESULTS:")
print("="*80)
print(f"Best trial: {study.best_trial.number}")
print(f"Best eval loss: {study.best_trial.value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*80)

# Save study
joblib.dump(study, '/content/drive/MyDrive/optuna_study_final.pkl')
print("\n✅ Optuna study saved to Google Drive!")

[I 2026-02-22 10:43:42,774] A new study created in memory with name: no-name-39a88cb7-a579-4159-a7a9-72e620ca8489


Starting Optuna hyperparameter search...
Trials: 10
Estimated time: ~3-4 hours on A100


Trial 0
LR: 5.39e-05, LoRA r: 16, alpha: 32
Batch size: 16, Warmup: 291



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
200,1.028416,1.089260
400,0.934124,0.942179


[I 2026-02-22 12:24:56,068] Trial 9 finished with value: 0.9020498394966125 and parameters: {'learning_rate': 4.881047730084562e-05, 'lora_r': 32, 'lora_alpha': 64, 'batch_size': 16, 'warmup_steps': 337}. Best is trial 1 with value: 0.8916620016098022.



OPTUNA RESULTS:
Best trial: 1
Best eval loss: 0.8917

Best hyperparameters:
  learning_rate: 0.00010136723607810614
  lora_r: 64
  lora_alpha: 64
  batch_size: 16
  warmup_steps: 238

✅ Optuna study saved to Google Drive!


## 7. Final Training with Best Hyperparameters

**A100 Optimized - Full Training**

In [22]:
print("\n" + "="*80)
print("FINAL TRAINING WITH BEST HYPERPARAMETERS")
print("="*80)

best_params = study.best_trial.params
print(f"\nUsing: {best_params}\n")

# Load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

final_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

final_tokenizer = AutoTokenizer.from_pretrained(model_name)
final_tokenizer.pad_token = final_tokenizer.eos_token
final_tokenizer.padding_side = "right"

# LoRA with best params
final_peft_config = LoraConfig(
    r=best_params["lora_r"],
    lora_alpha=best_params["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

final_model = prepare_model_for_kbit_training(final_model)

# Final training args - A100 OPTIMIZED FOR FULL TRAINING
final_training_args = TrainingArguments(
    output_dir="./adaptive-learning-final",
    num_train_epochs=3,  # Full 3 epochs
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    gradient_accumulation_steps=1,  # No accumulation with large batches
    learning_rate=best_params["learning_rate"],
    bf16=True,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    warmup_steps=best_params["warmup_steps"],
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    dataloader_num_workers=4,  # A100 + High RAM
    dataloader_pin_memory=True,
    group_by_length=True,
    fp16=False,  # Using bf16 instead
    report_to="none",
)

# Final trainer with FULL dataset
final_trainer = SFTTrainer(
    model=final_model,
    train_dataset=train_dataset,  # FULL training set
    eval_dataset=val_dataset,
    peft_config=final_peft_config,
    args=final_training_args,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=final_tokenizer,
)

print(f"Final training starting...")
print(f"Training examples: {len(train_dataset)}")
print(f"Batch size: {best_params['batch_size']}")
print(f"Estimated time: ~3-4 hours on A100\n")

final_trainer.train()

print("\n✅ Final training complete!")


FINAL TRAINING WITH BEST HYPERPARAMETERS

Using: {'learning_rate': 0.00010136723607810614, 'lora_r': 64, 'lora_alpha': 64, 'batch_size': 16, 'warmup_steps': 238}



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'dataset_text_field'

## 8. Save Final Model

In [24]:
output_dir = "/content/drive/MyDrive/adaptive-learning-evaluation-FINAL"

print(f"Saving model to: {output_dir}")
final_trainer.model.save_pretrained(output_dir)
final_tokenizer.save_pretrained(output_dir)

# Save training info
training_info = {
    "model": model_name,
    "datasets": ["SciQ", "GSM8K", "OpenBookQA", "ARC-Easy", "ARC-Challenge", "CommonsenseQA", "ELI5"],
    "total_train_examples": len(train_dataset),
    "total_val_examples": len(val_dataset),
    "best_hyperparameters": best_params,
    "best_eval_loss": study.best_trial.value,
    "optuna_trials": len(study.trials),
    "hardware": "A100 40GB",
    "training_epochs": 3,
}

with open(f'{output_dir}/training_info.json', 'w') as f:
    json.dump(training_info, f, indent=2)

print("\n" + "="*80)
print("FINAL MODEL SAVED!")
print("="*80)
print(f"Location: {output_dir}")
print(f"\nTraining info:")
print(json.dumps(training_info, indent=2))
print("="*80)

Saving model to: /content/drive/MyDrive/adaptive-learning-evaluation-FINAL

FINAL MODEL SAVED!
Location: /content/drive/MyDrive/adaptive-learning-evaluation-FINAL

Training info:
{
  "model": "mistralai/Mistral-7B-v0.1",
  "datasets": [
    "SciQ",
    "GSM8K",
    "OpenBookQA",
    "ARC-Easy",
    "ARC-Challenge",
    "CommonsenseQA",
    "ELI5"
  ],
  "total_train_examples": 37830,
  "total_val_examples": 5708,
  "best_hyperparameters": {
    "learning_rate": 0.00010136723607810614,
    "lora_r": 64,
    "lora_alpha": 64,
    "batch_size": 16,
    "warmup_steps": 238
  },
  "best_eval_loss": 0.8916620016098022,
  "optuna_trials": 10,
  "hardware": "A100 40GB",
  "training_epochs": 3
}


## 9. Test Final Model

In [25]:
# Test Science
test_sci = [sciq['test'][i] for i in range(2)]
prompt = create_multi_question_prompt_sciq(test_sci, True, "videos")
test_input = prompt.split("### Evaluation:")[0] + "### Evaluation:"

inputs = final_tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = final_trainer.model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

print("\n" + "="*80)
print("SCIENCE TEST (Correct Answers):")
print("="*80)
print(final_tokenizer.decode(outputs[0], skip_special_tokens=True))
print("="*80)

# Test Math
test_math = [gsm8k['test'][i] for i in range(2)]
prompt = create_multi_question_prompt_gsm8k(test_math, False, "articles")
test_input = prompt.split("### Evaluation:")[0] + "### Evaluation:"

inputs = final_tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = final_trainer.model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

print("\n" + "="*80)
print("MATH TEST (Wrong Answers):")
print("="*80)
print(final_tokenizer.decode(outputs[0], skip_special_tokens=True))
print("="*80)

print("\n✅ Testing complete!")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



SCIENCE TEST (Correct Answers):
### Domain: [SCIENCE]
### Context:
Oxidants and Reductants Compounds that are capable of accepting electrons, such as O 2 or F2, are calledoxidants (or oxidizing agents) because they can oxidize other compounds. In the process of accepting electrons, an oxidant is reduced. Compounds that are capable of donating electrons, such as sodium metal or cyclohexane (C6H12), are calledreductants (or reducing agents) because they can cause the reduction of another compound. In the process of donating electrons, a reductant is oxidized. These relationships are summarized in Equation 3.30: Equation 3.30 Saylor URL: http://www. saylor. org/books.

### Assessment:
Q1: Compounds that are capable of accepting electrons, such as o 2 or f2, are called what?
Student Answer: oxidants

Q2: What term in biotechnology means a genetically exact copy of an organism?
Student Answer: clone

### User Preference: videos

### Evaluation:
The student demonstrated strong understanding

---

## Summary

**A100-Optimized Training:**
- Batch size: 16-32 (vs standard 4-8)
- LoRA rank: 16-64 (vs standard 8-16)
- Workers: 4 (vs standard 0-2)
- Larger training set samples for Optuna

**Results:**
- Model: Mistral 7B with LoRA
- Datasets: 6 (50K+ examples)
- Optuna: 10 trials
- Training: 3 epochs
- Total time: ~6-7 hours

**Model saved to:** `/content/drive/MyDrive/adaptive-learning-evaluation-FINAL/`

# Task
SFTTrainer.__init__() no longer supports the `dataset_text_field` argument. I will remove this argument from the `SFTTrainer` call in the `objective` function to resolve the error.

Then, I will proceed with the rest of the plan, which includes restarting the Optuna optimization, performing final training, saving the final model to Google Drive, and testing the final model.

Finally, I will summarize the completion of the Optuna optimization, final training, and model saving, along with an overview of the results.

What would you like to do?

## Fix SFTTrainer Argument

### Subtask:
Modify the `objective` function in cell `WsguZDSy5PMO` to remove the `dataset_text_field='text'` argument from the `SFTTrainer` constructor, as it is no longer supported in newer `trl` versions. The `SFTTrainer` will default to using the 'text' column, which is already present in the prepared datasets.


**Reasoning**:
The subtask requires modifying the `objective` function to remove an unsupported argument from the `SFTTrainer` constructor. I will generate a `code_block` to update the content of cell `WsguZDSy5PMO` by removing `dataset_text_field="text"`.



In [13]:
model_name = "mistralai/Mistral-7B-v0.1"

def objective(trial):
    """A100-optimized Optuna objective"""

    # Hyperparameters - OPTIMIZED FOR A100
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    lora_r = trial.suggest_categorical("lora_r", [16, 32, 64])  # Larger ranks
    lora_alpha = trial.suggest_categorical("lora_alpha", [32, 64, 128])  # Larger alpha
    batch_size = trial.suggest_categorical("batch_size", [16, 32])  # MUCH larger batches for A100
    warmup_steps = trial.suggest_int("warmup_steps", 100, 400)

    print(f"\n{'='*80}")
    print(f"Trial {trial.number}")
    print(f"LR: {learning_rate:.2e}, LoRA r: {lora_r}, alpha: {lora_alpha}")
    print(f"Batch size: {batch_size}, Warmup: {warmup_steps}")
    print(f"{'='*80}\n")

    # Load model with 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # LoRA config
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = prepare_model_for_kbit_training(model)

    # Training args - A100 OPTIMIZED
    training_args = TrainingArguments(
        output_dir=f"./optuna-trial-{trial.number}",
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=1,  # No accumulation with large batches
        learning_rate=learning_rate,
        bf16=True,
        eval_strategy="steps",
        eval_steps=200,
        logging_steps=50,
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        max_grad_norm=0.3,
        save_strategy="no",
        dataloader_num_workers=4,  # A100 + High RAM = more workers
        dataloader_pin_memory=True,
        group_by_length=True,  # Faster training
    )

    # Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset.select(range(8000)),  # Larger subset for Optuna
        eval_dataset=val_dataset.select(range(800)),
        peft_config=peft_config,
        args=training_args,
        max_seq_length=1024,
        tokenizer=tokenizer,
    )

    # Train
    trainer.train()

    # Evaluate
    eval_result = trainer.evaluate()
    eval_loss = eval_result["eval_loss"]

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()

    return eval_loss

print("✅ Optuna objective function ready")

✅ Optuna objective function ready


## Restart Optuna Optimization

### Subtask:
Re-run the Optuna hyperparameter optimization in cell `qrJU6BJM5PMO`. This will execute 10 trials to find the best hyperparameters for the model training, using the corrected `SFTTrainer` call.


**Reasoning**:
The previous step corrected the `SFTTrainer` argument. Now, I will re-run the Optuna hyperparameter optimization by executing the code cell `qrJU6BJM5PMO`.



In [14]:
# Run Optuna
print("Starting Optuna hyperparameter search...")
print("Trials: 10")
print("Estimated time: ~3-4 hours on A100\n")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("\n" + "="*80)
print("OPTUNA RESULTS:")
print("="*80)
print(f"Best trial: {study.best_trial.number}")
print(f"Best eval loss: {study.best_trial.value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*80)

# Save study
joblib.dump(study, '/content/drive/MyDrive/optuna_study_final.pkl')
print("\n✅ Optuna study saved to Google Drive!")

[I 2026-02-22 10:37:26,438] A new study created in memory with name: no-name-a6c6914c-920a-4f33-b917-0f39f025094c


Starting Optuna hyperparameter search...
Trials: 10
Estimated time: ~3-4 hours on A100


Trial 0
LR: 2.02e-05, LoRA r: 64, alpha: 32
Batch size: 16, Warmup: 131



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[W 2026-02-22 10:37:33,574] Trial 0 failed with parameters: {'learning_rate': 2.0246674844595484e-05, 'lora_r': 64, 'lora_alpha': 32, 'batch_size': 16, 'warmup_steps': 131} because of the following error: TypeError("SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-4245725885.py", line 74, in objective
    trainer = SFTTrainer(
              ^^^^^^^^^^^
TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'
[W 2026-02-22 10:37:33,575] Trial 0 failed with value None.


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'max_seq_length'

**Reasoning**:
The previous attempt to run Optuna failed because the `SFTTrainer` no longer accepts the `max_seq_length` argument. I need to remove this argument from the `SFTTrainer` constructor in the `objective` function.



In [15]:
model_name = "mistralai/Mistral-7B-v0.1"

def objective(trial):
    """A100-optimized Optuna objective"""

    # Hyperparameters - OPTIMIZED FOR A100
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    lora_r = trial.suggest_categorical("lora_r", [16, 32, 64])  # Larger ranks
    lora_alpha = trial.suggest_categorical("lora_alpha", [32, 64, 128])  # Larger alpha
    batch_size = trial.suggest_categorical("batch_size", [16, 32])  # MUCH larger batches for A100
    warmup_steps = trial.suggest_int("warmup_steps", 100, 400)

    print(f"\n{'='*80}")
    print(f"Trial {trial.number}")
    print(f"LR: {learning_rate:.2e}, LoRA r: {lora_r}, alpha: {lora_alpha}")
    print(f"Batch size: {batch_size}, Warmup: {warmup_steps}")
    print(f"{'='*80}\n")

    # Load model with 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # LoRA config
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = prepare_model_for_kbit_training(model)

    # Training args - A100 OPTIMIZED
    training_args = TrainingArguments(
        output_dir=f"./optuna-trial-{trial.number}",
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=1,  # No accumulation with large batches
        learning_rate=learning_rate,
        bf16=True,
        eval_strategy="steps",
        eval_steps=200,
        logging_steps=50,
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        max_grad_norm=0.3,
        save_strategy="no",
        dataloader_num_workers=4,  # A100 + High RAM = more workers
        dataloader_pin_memory=True,
        group_by_length=True,  # Faster training
    )

    # Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset.select(range(8000)),  # Larger subset for Optuna
        eval_dataset=val_dataset.select(range(800)),
        peft_config=peft_config,
        args=training_args,
        tokenizer=tokenizer,
    )

    # Train
    trainer.train()

    # Evaluate
    eval_result = trainer.evaluate()
    eval_loss = eval_result["eval_loss"]

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()

    return eval_loss

print("✅ Optuna objective function ready")

✅ Optuna objective function ready


**Reasoning**:
The `objective` function has been updated to fix the `TypeError`. Now, I will re-run the Optuna hyperparameter optimization by executing the code cell `qrJU6BJM5PMO`.



In [16]:
# Run Optuna
print("Starting Optuna hyperparameter search...")
print("Trials: 10")
print("Estimated time: ~3-4 hours on A100\n")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("\n" + "="*80)
print("OPTUNA RESULTS:")
print("="*80)
print(f"Best trial: {study.best_trial.number}")
print(f"Best eval loss: {study.best_trial.value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*80)

# Save study
joblib.dump(study, '/content/drive/MyDrive/optuna_study_final.pkl')
print("\n✅ Optuna study saved to Google Drive!")

[I 2026-02-22 10:37:55,432] A new study created in memory with name: no-name-02be67df-c2db-4181-bbbd-d18249bac785


Starting Optuna hyperparameter search...
Trials: 10
Estimated time: ~3-4 hours on A100


Trial 0
LR: 4.41e-04, LoRA r: 16, alpha: 32
Batch size: 16, Warmup: 268



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[W 2026-02-22 10:38:01,975] Trial 0 failed with parameters: {'learning_rate': 0.0004405188497635218, 'lora_r': 16, 'lora_alpha': 32, 'batch_size': 16, 'warmup_steps': 268} because of the following error: TypeError("SFTTrainer.__init__() got an unexpected keyword argument 'tokenizer'").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-3392286093.py", line 74, in objective
    trainer = SFTTrainer(
              ^^^^^^^^^^^
TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'tokenizer'
[W 2026-02-22 10:38:01,977] Trial 0 failed with value None.


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'tokenizer'

**Reasoning**:
The previous attempt to run Optuna failed because the `SFTTrainer` no longer accepts the `tokenizer` argument directly in its constructor. I need to remove this argument from the `SFTTrainer` call within the `objective` function.



In [17]:
model_name = "mistralai/Mistral-7B-v0.1"

def objective(trial):
    """A100-optimized Optuna objective"""

    # Hyperparameters - OPTIMIZED FOR A100
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    lora_r = trial.suggest_categorical("lora_r", [16, 32, 64])  # Larger ranks
    lora_alpha = trial.suggest_categorical("lora_alpha", [32, 64, 128])  # Larger alpha
    batch_size = trial.suggest_categorical("batch_size", [16, 32])  # MUCH larger batches for A100
    warmup_steps = trial.suggest_int("warmup_steps", 100, 400)

    print(f"\n{'='*80}")
    print(f"Trial {trial.number}")
    print(f"LR: {learning_rate:.2e}, LoRA r: {lora_r}, alpha: {lora_alpha}")
    print(f"Batch size: {batch_size}, Warmup: {warmup_steps}")
    print(f"{'='*80}\n")

    # Load model with 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # LoRA config
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = prepare_model_for_kbit_training(model)

    # Training args - A100 OPTIMIZED
    training_args = TrainingArguments(
        output_dir=f"./optuna-trial-{trial.number}",
        num_train_epochs=1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=1,  # No accumulation with large batches
        learning_rate=learning_rate,
        bf16=True,
        eval_strategy="steps",
        eval_steps=200,
        logging_steps=50,
        warmup_steps=warmup_steps,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        max_grad_norm=0.3,
        save_strategy="no",
        dataloader_num_workers=4,  # A100 + High RAM = more workers
        dataloader_pin_memory=True,
        group_by_length=True,  # Faster training
    )

    # Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset.select(range(8000)),  # Larger subset for Optuna
        eval_dataset=val_dataset.select(range(800)),
        peft_config=peft_config,
        args=training_args
    )

    # Train
    trainer.train()

    # Evaluate
    eval_result = trainer.evaluate()
    eval_loss = eval_result["eval_loss"]

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()

    return eval_loss

print("✅ Optuna objective function ready")

✅ Optuna objective function ready


**Reasoning**:
The `objective` function has been updated to fix the `TypeError`. Now, I will re-run the Optuna hyperparameter optimization by executing the code cell `qrJU6BJM5PMO`.



In [18]:
# Run Optuna
print("Starting Optuna hyperparameter search...")
print("Trials: 10")
print("Estimated time: ~3-4 hours on A100\n")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("\n" + "="*80)
print("OPTUNA RESULTS:")
print("="*80)
print(f"Best trial: {study.best_trial.number}")
print(f"Best eval loss: {study.best_trial.value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")
print("="*80)

# Save study
joblib.dump(study, '/content/drive/MyDrive/optuna_study_final.pkl')
print("\n✅ Optuna study saved to Google Drive!")

[I 2026-02-22 10:38:24,436] A new study created in memory with name: no-name-9850d209-3ec7-4f7b-b3ef-0c6da91c8c62


Starting Optuna hyperparameter search...
Trials: 10
Estimated time: ~3-4 hours on A100


Trial 0
LR: 2.33e-05, LoRA r: 64, alpha: 32
Batch size: 16, Warmup: 234



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
200,1.168811,1.170576


[W 2026-02-22 10:43:39,191] Trial 0 failed with parameters: {'learning_rate': 2.328413604953684e-05, 'lora_r': 64, 'lora_alpha': 32, 'batch_size': 16, 'warmup_steps': 234} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-1627679597.py", line 83, in objective
    trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2174, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 2536, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py

KeyboardInterrupt: 

# Task
The previous attempt to run Optuna failed because the `SFTTrainer` no longer accepts the `tokenizer` argument directly in its constructor. I need to remove this argument from the `SFTTrainer` call within the `objective` function.

**Plan:**

*   **Fix SFTTrainer Argument in `objective` function**: Remove the `tokenizer=tokenizer` argument from the `SFTTrainer` constructor in the `objective` function (cell `6c9be0d8`).
*   **Restart Optuna Optimization**: Re-run the Optuna hyperparameter optimization in cell `qrJU6BJM5PMO`.
*   **Fix SFTTrainer Argument in Final Training**: Remove the `dataset_text_field='text'`, `max_seq_length`, and `tokenizer` arguments from the `SFTTrainer` constructor in cell `rWHltna95PMO`.
*   **Perform Final Training**: After fixing the `SFTTrainer` argument, re-run the final model training in cell `rWHltna95PMO` using the optimized parameters from the Optuna study. This step will train the model on the full dataset for 3 epochs.
*   **Save Final Model**: Save the fully trained model and tokenizer, along with the training information, to Google Drive by running cell `v4_3NygL5PMO`.
*   **Test Final Model**: Execute the testing cells `CT7md37b5PMP` to verify the functionality of the newly trained model on example Science and Math prompts.
*   **Final Task**: Summarize the completion of the final training and model saving, and provide an overview of the results.

## Fix SFTTrainer Argument in Final Training

### Subtask:
Remove the `dataset_text_field='text'`, `max_seq_length`, and `tokenizer` arguments from the `SFTTrainer` constructor in cell `rWHltna95PMO`. The `SFTTrainer` will default to using the 'text' column, which is already present in the prepared datasets.


**Reasoning**:
I need to remove the `dataset_text_field`, `max_seq_length`, and `tokenizer` arguments from the `SFTTrainer` constructor in cell `rWHltna95PMO` to resolve the `TypeError` encountered during the previous execution. This modification will align the code with the updated `trl` library API.



In [23]:
print("\n" + "="*80)
print("FINAL TRAINING WITH BEST HYPERPARAMETERS")
print("="*80)

best_params = study.best_trial.params
print(f"\nUsing: {best_params}\n")

# Load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

final_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

final_tokenizer = AutoTokenizer.from_pretrained(model_name)
final_tokenizer.pad_token = final_tokenizer.eos_token
final_tokenizer.padding_side = "right"

# LoRA with best params
final_peft_config = LoraConfig(
    r=best_params["lora_r"],
    lora_alpha=best_params["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

final_model = prepare_model_for_kbit_training(final_model)

# Final training args - A100 OPTIMIZED FOR FULL TRAINING
final_training_args = TrainingArguments(
    output_dir="./adaptive-learning-final",
    num_train_epochs=3,  # Full 3 epochs
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    gradient_accumulation_steps=1,  # No accumulation with large batches
    learning_rate=best_params["learning_rate"],
    bf16=True,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    warmup_steps=best_params["warmup_steps"],
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    dataloader_num_workers=4,  # A100 + High RAM
    dataloader_pin_memory=True,
    group_by_length=True,
    fp16=False,  # Using bf16 instead
    report_to="none",
)

# Final trainer with FULL dataset
final_trainer = SFTTrainer(
    model=final_model,
    train_dataset=train_dataset,  # FULL training set
    eval_dataset=val_dataset,
    peft_config=final_peft_config,
    args=final_training_args
)

print(f"Final training starting...")
print(f"Training examples: {len(train_dataset)}")
print(f"Batch size: {best_params['batch_size']}")
print(f"Estimated time: ~3-4 hours on A100\n")

final_trainer.train()

print("\n✅ Final training complete!")


FINAL TRAINING WITH BEST HYPERPARAMETERS

Using: {'learning_rate': 0.00010136723607810614, 'lora_r': 64, 'lora_alpha': 64, 'batch_size': 16, 'warmup_steps': 238}



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/37830 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/37830 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/37830 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/5708 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/5708 [00:00<?, ? examples/s]

Final training starting...
Training examples: 37830
Batch size: 16
Estimated time: ~3-4 hours on A100



Step,Training Loss,Validation Loss
500,0.910392,0.961884
1000,0.887078,0.927162
1500,0.848169,0.894969
2000,0.849556,0.889363
2500,0.721423,0.906413
3000,0.728536,0.907011
3500,0.703605,0.919662
4000,0.671783,0.929220
4500,0.673736,0.937261
5000,0.578476,1.002041



✅ Final training complete!
